In [1]:
import pandas as pd
import ast
from urllib.parse import urlparse

### Load both CSVs

In [10]:
df = pd.read_csv("data/df.csv")

### Source classification

Two dimensions per reference:
- **source_type**: institutional | academic | government | health_platform | lay | encyclopedia
- **locality**: western | chinese | global | neutral

In [6]:
# Domain → (source_type, locality)
DOMAIN_LOOKUP = {
    # --- Western institutional ---
    "nih.gov":             ("institutional", "western"),
    "ncbi.nlm.nih.gov":    ("institutional", "western"),
    "niddk.nih.gov":       ("institutional", "western"),
    "cancer.gov":          ("institutional", "western"),
    "cdc.gov":             ("institutional", "western"),
    "mayoclinic.org":      ("institutional", "western"),
    "clevelandclinic.org": ("institutional", "western"),
    "hopkinsmedicine.org": ("institutional", "western"),
    "heart.org":           ("institutional", "western"),
    "diabetes.org":        ("institutional", "western"),
    "cancer.org":          ("institutional", "western"),
    "lung.org":            ("institutional", "western"),
    "nami.org":            ("institutional", "western"),
    # --- Global institutional ---
    "who.int":             ("institutional", "global"),
    "unaids.org":          ("institutional", "global"),
    # --- Chinese institutional ---
    "nhc.gov.cn":          ("institutional", "chinese"),
    "chinacdc.cn":         ("institutional", "chinese"),
    "bjmu.edu.cn":         ("institutional", "chinese"),
    # --- Academic ---
    "pubmed.ncbi.nlm.nih.gov": ("academic", "western"),
    "thelancet.com":       ("academic", "western"),
    "nejm.org":            ("academic", "western"),
    "bmj.com":             ("academic", "western"),
    "nature.com":          ("academic", "western"),
    "cnki.net":            ("academic", "chinese"),
    "wanfangdata.com.cn":  ("academic", "chinese"),
    # --- Western health platforms ---
    "webmd.com":           ("health_platform", "western"),
    "healthline.com":      ("health_platform", "western"),
    "medicalnewstoday.com":("health_platform", "western"),
    "verywellhealth.com":  ("health_platform", "western"),
    "everydayhealth.com":  ("health_platform", "western"),
    "drugs.com":           ("health_platform", "western"),
    # --- Chinese health platforms ---
    "youlai.cn":           ("health_platform", "chinese"),
    "bohe.cn":             ("health_platform", "chinese"),
    "jiankang.baidu.com":  ("health_platform", "chinese"),
    "120.net":             ("health_platform", "chinese"),
    "dxy.cn":              ("health_platform", "chinese"),
    "xywy.com":            ("health_platform", "chinese"),
    # --- Encyclopedias ---
    "baike.baidu.com":     ("encyclopedia", "chinese"),
    "en.wikipedia.org":    ("encyclopedia", "western"),
    "zh.wikipedia.org":    ("encyclopedia", "chinese"),
    # --- Lay / commercial ---
    "youtube.com":         ("lay", "western"),
    "reddit.com":          ("lay", "western"),
    "zhihu.com":           ("lay", "chinese"),
    "haokan.baidu.com":    ("lay", "chinese"),  # Baidu video
}

def extract_domain(url: str) -> str:
    try:
        return urlparse(url).netloc.replace("www.", "").replace("m.", "")
    except:
        return ""

def classify_source(ref: dict) -> dict:
    """
    Classify a single reference dict by source_type and locality.
    Falls back to 'unknown' if domain not in lookup.
    """
    domain = extract_domain(ref.get("link", ""))

    # Try exact domain match first, then suffix match
    match = DOMAIN_LOOKUP.get(domain)
    if not match:
        for known_domain, classification in DOMAIN_LOOKUP.items():
            if domain.endswith(known_domain):
                match = classification
                break

    source_type, locality = match if match else ("unknown", "unknown")

    return {
        "title":       ref.get("title", ""),
        "link":        ref.get("link", ""),
        "source":      ref.get("source", ""),
        "domain":      domain,
        "source_type": source_type,
        "locality":    locality,
    }

### Explode references into a flat table

One row per reference, with its parent query and platform attached.
This is the primary analysis table.

In [7]:
def explode_references(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in df.iterrows():
        refs = row.get("all_references")
        if not isinstance(refs, list) or len(refs) == 0:
            continue
        for ref in refs:
            classified = classify_source(ref)
            classified.update({
                "query_en": row["query_en"],
                "platform": row["platform"],
                "topic":    row["topic"],
                "section":  row["section"],
            })
            rows.append(classified)
    return pd.DataFrame(rows)

google_refs = explode_references(google_df)
baidu_refs  = explode_references(baidu_df)
all_refs    = pd.concat([google_refs, baidu_refs], ignore_index=True)

print(f"Google refs: {len(google_refs)} | Baidu refs: {len(baidu_refs)}")
all_refs.head()

Google refs: 998 | Baidu refs: 367


,title,link,source,domain,source_type,locality,query_en,platform,topic,section
0,Breast cancer - Symptoms and causes - Mayo Clinic,https://www.mayoclinic.org/diseases-conditions...,Mayo Clinic,mayoclinic.org,institutional,western,breast cancer,google,cancer,common_conditions
1,Breast cancer - World Health Organization (WHO),https://www.who.int/news-room/fact-sheets/deta...,World Health Organization (WHO),who.int,institutional,global,breast cancer,google,cancer,common_conditions
2,Breast Cancer Statistics | How Common Is Breas...,https://www.cancer.org/cancer/types/breast-can...,American Cancer Society,cancer.org,institutional,western,breast cancer,google,cancer,common_conditions
3,Breast Cancer Basics - CDC,https://www.cdc.gov/breast-cancer/about/index....,Centers for Disease Control and Prevention | C...,cdc.gov,institutional,western,breast cancer,google,cancer,common_conditions
4,Breast Cancer Prevention - NCI,https://www.cancer.gov/types/breast/causes-ris...,National Cancer Institute (.gov),cancer.gov,institutional,western,breast cancer,google,cancer,common_conditions


### Review unknowns

Check which domains weren't classified — add them to DOMAIN_LOOKUP above and re-run.

In [8]:
unknowns = all_refs[all_refs["source_type"] == "unknown"]["domain"].value_counts()
print(f"Unknown domains ({len(unknowns)} unique):")
print(unknowns.head(30))

Unknown domains (417 unique):
domain
mfk.com                         25
facebook.com                    24
medlineplus.gov                 23
hiv.gov                         15
google.com                      10
mp.weixin.qq.com                 7
health.harvard.edu               7
yalemedicine.org                 7
plannedparenthood.org            7
goodrx.com                       7
sciencedirect.com                7
pennmedicine.org                 6
mountsinai.org                   6
nhs.uk                           5
mdanderson.org                   5
fda.gov                          5
hrc.org                          4
betterhealth.vic.gov.au          4
ucsfhealth.org                   4
samhsa.gov                       4
instagracom                      4
bannerhealth.com                 3
recovercovid.org                 3
cityofhope.org                   3
uofmhealth.org                   3
rush.edu                         3
associatesinnephrologypc.com     3
medinfo.co.uk     

### Add summary columns back to main dfs

In [9]:
def add_ref_summary(df: pd.DataFrame, refs_df: pd.DataFrame) -> pd.DataFrame:
    summary = refs_df.groupby("query_en").agg(
        num_refs          = ("link", "count"),
        num_institutional = ("source_type", lambda x: (x == "institutional").sum()),
        num_academic      = ("source_type", lambda x: (x == "academic").sum()),
        num_health_platform=("source_type", lambda x: (x == "health_platform").sum()),
        num_encyclopedia  = ("source_type", lambda x: (x == "encyclopedia").sum()),
        num_lay           = ("source_type", lambda x: (x == "lay").sum()),
        num_unknown       = ("source_type", lambda x: (x == "unknown").sum()),
        num_western       = ("locality",    lambda x: (x == "western").sum()),
        num_chinese       = ("locality",    lambda x: (x == "chinese").sum()),
        num_global        = ("locality",    lambda x: (x == "global").sum()),
    ).reset_index()
    return df.merge(summary, on="query_en", how="left")

google_df = add_ref_summary(google_df, google_refs)
baidu_df  = add_ref_summary(baidu_df,  baidu_refs)

google_df.head()

,Unnamed: 0,location,query_en,has_ai_overview,all_items,cited_sources,all_references,paa_questions,platform,topic,...,num_refs,num_institutional,num_academic,num_health_platform,num_encyclopedia,num_lay,num_unknown,num_western,num_chinese,num_global
0,0,United States,breast cancer,True,[A new lump or mass in the breast or underarm ...,[{'title': 'Breast cancer - World Health Organ...,[{'title': 'Breast cancer - Symptoms and cause...,"['What are the top 3 signs of breast cancer?',...",google,cancer,...,14.0,7.0,0.0,1.0,0.0,2.0,4.0,9.0,0.0,1.0
1,1,United States,what is cancer,True,[Tumor Formation: Extra cells form masses of t...,"[{'title': 'What Is Cancer? Symptoms, Causes &...",[{'title': 'What Is Cancer? - NCI - National C...,"['What is the simple definition of cancer?', '...",google,cancer,...,9.0,3.0,0.0,0.0,0.0,0.0,6.0,2.0,0.0,1.0
2,2,United States,cancer symptoms,True,[Unexplained Weight Loss: Losing $\ge 10$ poun...,[{'title': 'Cancer - Symptoms and causes - May...,[{'title': 'Cancer - Symptoms and causes - May...,"['What are the 7 warning signs for cancer?', '...",google,cancer,...,11.0,4.0,0.0,1.0,0.0,0.0,6.0,5.0,0.0,0.0
3,3,United States,lung cancer,True,[Non-Small Cell Lung Cancer (NSCLC): Accounts ...,[{'title': 'Lung cancer - World Health Organiz...,[{'title': 'What Is Lung Cancer? | Types of Lu...,['How long can you live with one lung cancer?'...,google,cancer,...,15.0,3.0,0.0,0.0,0.0,2.0,10.0,4.0,0.0,1.0
4,4,United States,skin cancer,True,[Basal Cell Carcinoma (BCC): The most common t...,"[{'title': 'Skin Cancer Basics - CDC', 'link':...",[{'title': 'Skin cancer - Symptoms and causes ...,['What does skin cancer look like when it just...,google,cancer,...,14.0,6.0,0.0,0.0,0.0,2.0,6.0,8.0,0.0,0.0


### Save

In [ ]:
google_df.to_csv("google_classified.csv", index=False)
baidu_df.to_csv("baidu_classified.csv",   index=False, encoding="utf-8-sig")
all_refs.to_csv("all_refs_classified.csv", index=False, encoding="utf-8-sig")
print("Saved: google_classified.csv, baidu_classified.csv, all_refs_classified.csv")